# MS-MARCO Abstractive QA Notebook for RAG

This notebook is separate from `main.ipynb` so your stable Natural Questions / Open-Domain QA work stays untouched.

Goal:
- Prepare MS-MARCO NLG-style data.
- Train a separate RAG model for abstractive QA using your stable 50k retriever setup.
- Evaluate with BLEU-1 and ROUGE-L, which are the metrics reported for MS-MARCO in the RAG paper.

Recommended output folder:

```text
outputs/rag_baseline_msmarco_50k
```


In [ ]:
# Cell 1 — install packages, mount Drive, and enter project folder

!pip install -q transformers accelerate faiss-cpu sentencepiece datasets pandas pyarrow tqdm

import os
import glob
import json
from pathlib import Path

from google.colab import drive

# Always start from /content so Drive remounts cleanly.
os.chdir("/content")

try:
    drive.flush_and_unmount()
except Exception:
    pass

drive.mount("/content/gdrive", force_remount=True)

# Change this only if your project folder is different.
PROJECT_DIR = "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj"

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f"PROJECT_DIR does not exist: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)

print("Now working in:", os.getcwd())
print("PROJECT_DIR:", PROJECT_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.4 MB/s eta 0:00:00
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/gdrive
Now working in: /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj
PROJECT_DIR: /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj


In [ ]:
# Cell 2 — verify GPU, RAM, disk, and key project files

import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDisk space:")
!df -h

print("\nRequired files:")
for fname in ["train.py", "custom_retriever.py", "config.py", "dataset.py", "checkpoint_utils.py"]:
    print(fname, "✅" if os.path.exists(fname) else "❌ missing")

print("\nCurrent data folder:")
!ls -lh data 2>/dev/null || true


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB

Disk space:
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   44G  193G  19% /
tmpfs            64M     0   64M   0% /dev
shm              83G     0   83G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
tmpfs            84G  236K   84G   1% /var/colab
/dev/sda1       242G   49G  194G  20% /kaggle/input
/dev/nvme0n1    369G   28K  350G   1% /mnt/local-scratch
tmpfs            84G     0   84G   0% /proc/acpi
tmpfs            84G     0   84G   0% /proc/scsi
tmpfs            84G     0   84G   0% /sys/firmware
drive           236G   53G  183G  23% /content/gdrive

Required files:
train.py ✅
custom_retriever.py ✅
config.py ✅
dataset.py ✅
checkpoint_utils.py ✅

Current data folder:
total 12M
-rw------- 1 root root 175K May  8 05:54 msmarco_dev.jsonl
-rw------- 1 root root 3.1M May  8 05:54 msmarco_train.jsonl
-rw------- 1 root root 386K May  4 09:05 nq_dev.jsonl
-rw------- 1 root root 8.3M May  4 

In [ ]:
# Cell 3 — make /content/data point to this project's data folder
#
# Your config.py may expect DATA_DIR = /content/data.
# This symlink makes /content/data use PROJECT_DIR/data.

if not os.path.exists("data"):
    os.makedirs("data", exist_ok=True)

!rm -rf /content/data
!ln -s "$PWD/data" /content/data

print("/content/data now points to:")
!ls -ld /content/data


/content/data now points to:
lrwxrwxrwx 1 root root 89 May  8 07:32 /content/data -> /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj/data


In [ ]:
# Cell 4 — make sure the old stable 50k retriever/index is available locally
#
# Your old train.py likely expects:
#   /content/wiki_dpr_built/wiki_dpr_nq.faiss
#   /content/wiki_dpr_built/passages.parquet
#
# This cell tries to find the old 50k FAISS index in Drive/project storage and copy it locally.
# It intentionally ignores the 60% spread index because this notebook uses the stable 50k setup.

import os
import glob
import shutil
from pathlib import Path

LOCAL_50K_INDEX_DIR = "/content/wiki_dpr_built"
local_index = os.path.join(LOCAL_50K_INDEX_DIR, "wiki_dpr_nq.faiss")
local_passages = os.path.join(LOCAL_50K_INDEX_DIR, "passages.parquet")

os.makedirs(LOCAL_50K_INDEX_DIR, exist_ok=True)

if os.path.exists(local_index) and os.path.exists(local_passages):
    print("Local 50k index already exists:", LOCAL_50K_INDEX_DIR)
else:
    print("Searching project folder for old 50k index files...")
    faiss_files = [
        f for f in glob.glob(os.path.join(PROJECT_DIR, "**", "*.faiss"), recursive=True)
        if "60pct" not in f and "spread" not in f
    ]

    print("Candidate FAISS files:")
    for f in faiss_files[:20]:
        print(" -", f)

    copied = False
    for f in faiss_files:
        d = os.path.dirname(f)
        p = os.path.join(d, "passages.parquet")
        if os.path.exists(p):
            print("\nUsing candidate index folder:", d)
            print("Copying to:", LOCAL_50K_INDEX_DIR)
            !rsync -ah --info=progress2 "$d/" "$LOCAL_50K_INDEX_DIR/"
            copied = True
            break

    if not copied:
        print("\nCould not automatically find the old 50k index.")
        print("If your old main notebook has cells that build/copy /content/wiki_dpr_built, run those first.")

print("\nFinal local index check:")
!ls -lh /content/wiki_dpr_built 2>/dev/null || true


Searching project folder for old 50k index files...
Candidate FAISS files:
 - /content/gdrive/MyDrive/JuniorYear/CS4782/final-proj/data/small_nq_index/index.faiss

Could not automatically find the old 50k index.
If your old main notebook has cells that build/copy /content/wiki_dpr_built, run those first.

Final local index check:
total 0


In [ ]:
import os
from google.colab import drive

# Get out of the broken Drive path first
os.chdir("/content")

# Remount Drive cleanly
try:
    drive.flush_and_unmount()
except Exception as e:
    print("Unmount warning:", e)

drive.mount("/content/gdrive", force_remount=True)

PROJECT_DIR = "/content/gdrive/MyDrive/JuniorYear/CS4782/final-proj"

# Fallback if Drive uses the shortcut target path
if not os.path.exists(PROJECT_DIR):
    PROJECT_DIR = "/content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj"

os.chdir(PROJECT_DIR)

print("Now working in:", os.getcwd())

Mounted at /content/gdrive
Now working in: /content/gdrive/.shortcut-targets-by-id/1o8yF586YhxDQB4V3dTdjn4u47_bCq-qW/final-proj


In [ ]:
!ls -lh data/small_nq_index
!find data/small_nq_index -maxdepth 2 -type f -print

total 147M
-rw------- 1 root root 147M May  4 12:47 index.faiss
drwx------ 2 root root 4.0K May  4 12:47 passages
data/small_nq_index/index.faiss
data/small_nq_index/passages/data-00000-of-00001.arrow
data/small_nq_index/passages/state.json
data/small_nq_index/passages/dataset_info.json


In [ ]:
# Cell 5 — write prepare_msmarco.py
#
# This script downloads MS-MARCO v2.1 from HuggingFace and converts it into JSONL files
# compatible with your existing dataset.py pattern:
#   {"question": "...", "answers": ["full sentence answer"]}
#
# For abstractive QA, we prefer wellFormedAnswers because those are full-sentence answers.

%%writefile prepare_msmarco.py
import argparse
import json
import os
from datasets import load_dataset


BAD_ANSWERS = {"", "[]", "No Answer Present", "No answer present."}


def is_no_answer(text):
    t = str(text).strip().lower()
    t = t.replace(".", "").replace("[", "").replace("]", "").strip()
    return (
        t == ""
        or t == "no answer present"
        or t == "no answer"
        or t == "none"
    )


def clean_answers(x):
    if x is None:
        return []

    if isinstance(x, str):
        x = x.strip()
        if is_no_answer(x):
            return []
        return [x]

    if isinstance(x, (list, tuple)):
        out = []
        for a in x:
            a = str(a).strip()
            if not is_no_answer(a):
                out.append(a)
        return out

    return []


def convert_split(ds, out_path, max_examples=None):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    kept = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for ex in ds:
            question = ex.get("query") or ex.get("question")
            if not question:
                continue

            # MS-MARCO NLG-style full sentence answers.
            answers = clean_answers(ex.get("wellFormedAnswers"))

            # Fallback if wellFormedAnswers is unavailable/empty.
            if not answers:
                answers = clean_answers(ex.get("answers"))

            if not answers:
                continue

            row = {
                "question": question,
                "answers": answers,
            }

            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            kept += 1

            if max_examples is not None and kept >= max_examples:
                break

    print(f"Wrote {kept:,} examples to {out_path}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_train", type=int, default=20000)
    parser.add_argument("--max_dev", type=int, default=1000)
    args = parser.parse_args()

    print("Loading MS-MARCO v2.1 from HuggingFace...")
    ds = load_dataset("ms_marco", "v2.1", trust_remote_code=True)
    print(ds)

    convert_split(ds["train"], "data/msmarco_train.jsonl", max_examples=args.max_train)
    convert_split(ds["validation"], "data/msmarco_dev.jsonl", max_examples=args.max_dev)


if __name__ == "__main__":
    main()


Overwriting prepare_msmarco.py


In [ ]:
# Cell 6 — prepare MS-MARCO data
#
# For a quick smoke test, use --max_train 5000 --max_dev 500.
# For a stronger result, use --max_train 20000 --max_dev 1000.

!python prepare_msmarco.py --max_train 20000 --max_dev 1000

print("\nPrepared files:")
!ls -lh data/msmarco_train.jsonl data/msmarco_dev.jsonl

print("\nFirst training example:")
!head -n 1 data/msmarco_train.jsonl


Loading MS-MARCO v2.1 from HuggingFace...
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ms_marco' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 101093
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 808731
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 101092
    })
})
Wrote 20,000 examples to data/msmarco_train.jsonl
Wrote 1,000 examples to data/msmarco_dev.jsonl

Prepared files:
-rw------- 1 root root 175K May  8 05:03 data/msmarco_dev.jsonl
-

In [ ]:
!grep -i "No Answer Present" data/msmarco_train.jsonl | head
!grep -i "No Answer Present" data/msmarco_dev.jsonl | head

In [ ]:
# Cell 7 — create a separate train_msmarco.py so train.py stays untouched
#
# This copies your stable old train.py, then patches only the dataset choices so
# --dataset msmarco is accepted.

from pathlib import Path
import re

src = Path("train.py")
dst = Path("train_msmarco.py")

if not src.exists():
    raise FileNotFoundError("train.py not found in the current project folder.")

text = src.read_text()

# Add msmarco to argparse dataset choices. Handles a few spacing variants.
text_new = re.sub(
    r'choices=\[\s*"nq"\s*,\s*"trivia"\s*,\s*"wq"\s*\]',
    'choices=["nq", "trivia", "wq", "msmarco"]',
    text,
)

if text_new == text and "msmarco" not in text:
    print("Warning: could not find the exact choices=[...] pattern to patch.")
    print("Open train_msmarco.py and manually add 'msmarco' to the dataset choices if needed.")

# Optional: avoid accidentally using an old output folder name if args.output_dir is omitted.
text_new = text_new.replace(
    'f"rag_baseline_{args.dataset}"',
    'f"rag_baseline_{args.dataset}"',
)

dst.write_text(text_new)

print("Created:", dst)
!grep -n "dataset" train_msmarco.py | head -8


Created: train_msmarco.py
23:from dataset import make_dataloader, exact_match_score
40:    p.add_argument("--dataset",    default="nq", choices=["nq", "trivia", "wq", "msmarco"])
63:    """Load model with our custom FAISS-based retriever (no load_dataset)."""
151:        args.output_dir = os.path.join(OUTPUT_DIR, f"rag_baseline_{args.dataset}")
174:    train_path = os.path.join(DATA_DIR, f"{args.dataset}_train.jsonl")
175:    dev_path   = os.path.join(DATA_DIR, f"{args.dataset}_dev.jsonl")


In [ ]:
# Cell 8 — sanity check train_msmarco.py can parse --dataset msmarco
#
# This should print the script help and include msmarco in the dataset choices.

!python train_msmarco.py --help | head -40


usage: train_msmarco.py [-h] [--dataset {nq,trivia,wq,msmarco}]
                        [--output_dir OUTPUT_DIR] [--n_docs N_DOCS]
                        [--epochs EPOCHS] [--batch_size BATCH_SIZE]
                        [--grad_accum GRAD_ACCUM] [--lr LR] [--fp16]
                        [--seed SEED] [--no_resume] [--eval_only]
                        [--max_train_examples MAX_TRAIN_EXAMPLES]
                        [--eval_steps EVAL_STEPS] [--save_steps SAVE_STEPS]
                        [--logging_steps LOGGING_STEPS]
                        [--keep_last_n_ckpts KEEP_LAST_N_CKPTS]

options:
  -h, --help            show this help message and exit
  --dataset {nq,trivia,wq,msmarco}
  --output_dir OUTPUT_DIR
  --n_docs N_DOCS
  --epochs EPOCHS
  --batch_size BATCH_SIZE
  --grad_accum GRAD_ACCUM
  --lr LR
  --fp16
  --seed SEED
  --no_resume           Start training from scratch even if checkpoints exist
  --eval_only           Skip training; just evaluate the latest checkpoint
  

In [ ]:
#Patch to because of wrong index path. This will help make cell 9 run successfully

from pathlib import Path

path = Path("train_msmarco.py")
text = path.read_text()

start = text.index("def load_model_and_tokenizer")
end = text.index("# ── Eval", start)

new_func = r'''
def load_model_and_tokenizer(model_path: str):
    """Load model with the stable old 50k custom FAISS retriever."""
    from transformers import RagRetriever

    log.info(f"Loading tokenizer: {model_path}")
    tokenizer = RagTokenizer.from_pretrained(model_path)

    index_path = "data/small_nq_index/index.faiss"
    passages_path = "data/small_nq_index/passages"

    if not os.path.exists(index_path):
        raise FileNotFoundError(f"Missing {index_path}")

    if not os.path.exists(passages_path):
        raise FileNotFoundError(f"Missing {passages_path}")

    log.info("Loading 50k custom retriever...")
    log.info(f"index_path = {index_path}")
    log.info(f"passages_path = {passages_path}")

    retriever = RagRetriever.from_pretrained(
        RAG_MODEL_NAME,
        index_name="custom",
        passages_path=passages_path,
        index_path=index_path,
    )

    log.info(f"Loading model: {model_path}")
    model = RagSequenceForGeneration.from_pretrained(
        model_path,
        retriever=retriever,
    )

    if hasattr(model.config, "num_return_sequences"):
        model.config.num_return_sequences = 1

    if hasattr(model, "generation_config"):
        model.generation_config.num_return_sequences = 1

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    log.info(f"Trainable parameters: {trainable:,} / {total:,}")

    return model, tokenizer

'''

text = text[:start] + new_func + "\n" + text[end:]
path.write_text(text)

print("Patched train_msmarco.py to use data/small_nq_index/")

Patched train_msmarco.py to use data/small_nq_index/


In [ ]:
#Patch to make make the loss a scalar before backprop for cell 9

from pathlib import Path

path = Path("train_msmarco.py")
text = path.read_text()

old = """                loss = outputs.loss
                accelerator.backward(loss)
"""

new = """                loss = outputs.loss

                # Some RAG/Transformers versions return a vector loss.
                # Backprop needs a scalar.
                if loss.ndim > 0:
                    loss = loss.mean()

                accelerator.backward(loss)
"""

if old not in text:
    raise ValueError("Could not find the loss block to patch.")

path.write_text(text.replace(old, new))

print("Patched train_msmarco.py so loss is scalar.")

Patched train_msmarco.py so loss is scalar.


In [ ]:
# Cell 9 — train RAG on MS-MARCO with the old stable 50k retriever
#
# Notes:
# - This is a separate run from your NQ model.
# - Built-in eval in train.py uses EM, which is not the MS-MARCO paper metric.
# - We set eval_steps very high so we evaluate BLEU-1/ROUGE-L separately later.
# - save_steps=500 gives you checkpoints even if Colab disconnects.

!python -u train_msmarco.py \
  --dataset msmarco \
  --fp16 \
  --epochs 1 \
  --max_train_examples 20000 \
  --eval_steps 100000 \
  --save_steps 1000 \
  --logging_steps 50 \
  --output_dir outputs/rag_baseline_msmarco_50k_answerable \
  --no_resume


2026-05-08 05:07:10,794 [INFO] Run state: {'latest_step': None, 'latest_ckpt': None, 'best_exists': False, 'num_checkpoints': 0}
2026-05-08 05:07:10,817 [INFO] Loading tokenizer: facebook/rag-sequence-nq
2026-05-08 05:07:10,999 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 05:07:11,008 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/rag-sequence-nq/c0d9c6ceda8a69c78091abb7aa734a97b75b89fd/config.json "HTTP/1.1 200 OK"
2026-05-08 05:07:11,115 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/question_encoder_tokenizer/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 05:07:11,123 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/rag-sequence-nq/c0d9c6ceda8a69c78091abb7aa734a97b75b89fd/question_encoder_tokenizer%2Ftokenizer_config.json "HTTP/1.1 200 OK"
2026-05-08 05:07:1

In [ ]:
#Cell 9 - 3 epochs

!python -u train_msmarco.py \
  --dataset msmarco \
  --fp16 \
  --epochs 3 \
  --max_train_examples 20000 \
  --eval_steps 100000 \
  --save_steps 1000 \
  --logging_steps 50 \
  --output_dir outputs/rag_baseline_msmarco_50k_answerable_3ep \
  --no_resume



2026-05-08 07:45:26,040 [INFO] Run state: {'latest_step': None, 'latest_ckpt': None, 'best_exists': False, 'num_checkpoints': 0}
2026-05-08 07:45:26,063 [INFO] Loading tokenizer: facebook/rag-sequence-nq
2026-05-08 07:45:26,376 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 07:45:26,376 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-08 07:45:26,382 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/rag-sequence-nq/c0d9c6ceda8a69c78091abb7aa734a97b75b89fd/config.json "HTTP/1.1 200 OK"
2026-05-08 07:45:26,389 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/rag-sequence-nq/c0d9c6ceda8a69c78091abb7aa734a97b75b89fd/config.json "HTTP/1.1 200 OK"
config.json: 4.60kB [00:00, 13.1MB/s]
2026-05-08 07:45:26,633 [INFO] HTTP Reques

In [ ]:
# Cell 10 — choose cleaned MS-MARCO checkpoint

from pathlib import Path
import re

OUTPUT_DIR = Path("outputs/rag_baseline_msmarco_50k_answerable")

best_dir = OUTPUT_DIR / "best"

ckpts = []
for p in OUTPUT_DIR.glob("checkpoint-*"):
    m = re.search(r"checkpoint-(\d+)$", str(p))
    if m:
        ckpts.append((int(m.group(1)), p))

ckpts = sorted(ckpts)

print("Available checkpoints:")
for step, path in ckpts:
    print(step, path)

if best_dir.exists():
    LATEST_MSMARCO_MODEL_DIR = str(best_dir)
elif ckpts:
    LATEST_MSMARCO_MODEL_DIR = str(ckpts[-1][1])
else:
    raise FileNotFoundError("No best model or checkpoint found.")

print("\nUsing model dir for BLEU/ROUGE:", LATEST_MSMARCO_MODEL_DIR)


Available checkpoints:
1000 outputs/rag_baseline_msmarco_50k_answerable/checkpoint-1000

Using model dir for BLEU/ROUGE: outputs/rag_baseline_msmarco_50k_answerable/checkpoint-1000


In [ ]:
# Cell 10 - 3 epochs

from pathlib import Path
import re

OUTPUT_DIR = Path("outputs/rag_baseline_msmarco_50k_answerable_3ep")

best_dir = OUTPUT_DIR / "best"

ckpts = []
for p in OUTPUT_DIR.glob("checkpoint-*"):
    m = re.search(r"checkpoint-(\d+)$", str(p))
    if m:
        ckpts.append((int(m.group(1)), p))

ckpts = sorted(ckpts)

print("Available checkpoints:")
for step, path in ckpts:
    print(step, path)

if best_dir.exists():
    LATEST_MSMARCO_MODEL_DIR = str(best_dir)
elif ckpts:
    LATEST_MSMARCO_MODEL_DIR = str(ckpts[-1][1])
else:
    raise FileNotFoundError("No best model or checkpoint found.")

print("\nUsing model dir for BLEU/ROUGE:", LATEST_MSMARCO_MODEL_DIR)


Available checkpoints:
2000 outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-2000
3000 outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-3000

Using model dir for BLEU/ROUGE: outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-3000


In [ ]:
# Cell 11 — write eval_msmarco_metrics.py
#
# This evaluates MS-MARCO with BLEU-1 and ROUGE-L.
# These are the metrics used for the MS-MARCO abstractive QA section of the RAG paper.

%%writefile eval_msmarco_metrics.py
import argparse
import json
import math
import os
import re
from collections import Counter

import torch
from tqdm import tqdm

from train_msmarco import load_model_and_tokenizer


def normalize_tokens(text):
    return re.findall(r"\w+", str(text).lower())


def lcs_len(a, b):
    """Longest common subsequence length for ROUGE-L."""
    dp = [0] * (len(b) + 1)

    for x in a:
        prev = 0
        for j, y in enumerate(b, start=1):
            temp = dp[j]
            if x == y:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp

    return dp[-1]


def rouge_l_f1(pred, ref):
    pred_toks = normalize_tokens(pred)
    ref_toks = normalize_tokens(ref)

    if not pred_toks or not ref_toks:
        return 0.0

    lcs = lcs_len(pred_toks, ref_toks)
    precision = lcs / len(pred_toks)
    recall = lcs / len(ref_toks)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def corpus_bleu1(preds, refs_list):
    """Corpus BLEU-1 with clipped unigram precision and brevity penalty."""
    clipped_total = 0
    pred_total = 0
    pred_len_total = 0
    ref_len_total = 0

    for pred, refs in zip(preds, refs_list):
        pred_toks = normalize_tokens(pred)
        ref_tokens_list = [normalize_tokens(r) for r in refs if str(r).strip()]

        if not pred_toks or not ref_tokens_list:
            continue

        pred_counts = Counter(pred_toks)

        best_ref_counts = Counter()
        best_overlap = -1
        best_ref_len = len(ref_tokens_list[0])

        for ref_toks in ref_tokens_list:
            ref_counts = Counter(ref_toks)
            overlap = sum((pred_counts & ref_counts).values())
            if overlap > best_overlap:
                best_overlap = overlap
                best_ref_counts = ref_counts
                best_ref_len = len(ref_toks)

        clipped_total += sum((pred_counts & best_ref_counts).values())
        pred_total += len(pred_toks)
        pred_len_total += len(pred_toks)
        ref_len_total += best_ref_len

    if pred_total == 0:
        return 0.0

    precision = clipped_total / pred_total

    if pred_len_total == 0:
        bp = 0.0
    elif pred_len_total > ref_len_total:
        bp = 1.0
    else:
        bp = math.exp(1 - ref_len_total / pred_len_total)

    return 100 * bp * precision


def load_jsonl(path, max_examples=None):
    examples = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))
            if max_examples is not None and len(examples) >= max_examples:
                break

    return examples


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_dir", required=True)
    parser.add_argument("--data_path", default="data/msmarco_dev.jsonl")
    parser.add_argument("--max_examples", type=int, default=500)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--n_docs", type=int, default=5)
    parser.add_argument("--num_beams", type=int, default=4)
    parser.add_argument("--max_new_tokens", type=int, default=64)
    parser.add_argument("--out_path", default="outputs/msmarco_predictions.jsonl")
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Loading model:", args.model_dir)
    model, tokenizer = load_model_and_tokenizer(args.model_dir)

    model.to(device)
    model.eval()

    examples = load_jsonl(args.data_path, max_examples=args.max_examples)

    preds = []
    refs_list = []

    os.makedirs(os.path.dirname(args.out_path), exist_ok=True)

    with open(args.out_path, "w", encoding="utf-8") as out_f:
        for start in tqdm(range(0, len(examples), args.batch_size)):
            batch = examples[start:start + args.batch_size]
            questions = [ex["question"] for ex in batch]

            enc = tokenizer.question_encoder(
                questions,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            )

            enc = {k: v.to(device) for k, v in enc.items()}

            with torch.no_grad():
                generated = model.generate(
                    input_ids=enc["input_ids"],
                    attention_mask=enc.get("attention_mask"),
                    n_docs=args.n_docs,
                    num_beams=args.num_beams,
                    num_return_sequences=1,
                    max_new_tokens=args.max_new_tokens,
                )

            batch_preds = tokenizer.batch_decode(generated, skip_special_tokens=True)

            for ex, pred in zip(batch, batch_preds):
                refs = ex["answers"]

                preds.append(pred)
                refs_list.append(refs)

                out_f.write(json.dumps({
                    "question": ex["question"],
                    "prediction": pred,
                    "references": refs,
                }, ensure_ascii=False) + "\n")

    rouge_l_scores = []
    for pred, refs in zip(preds, refs_list):
        best = max(rouge_l_f1(pred, ref) for ref in refs)
        rouge_l_scores.append(best)

    rouge_l = 100 * sum(rouge_l_scores) / len(rouge_l_scores)
    bleu1 = corpus_bleu1(preds, refs_list)

    print("=" * 80)
    print(f"Examples evaluated: {len(preds)}")
    print(f"ROUGE-L: {rouge_l:.2f}")
    print(f"BLEU-1:  {bleu1:.2f}")
    print(f"Predictions saved to: {args.out_path}")


if __name__ == "__main__":
    main()


Overwriting eval_msmarco_metrics.py


In [ ]:
# Cell 12 — evaluate the latest MS-MARCO checkpoint with BLEU-1 and ROUGE-L
#
# If this cell says LATEST_MSMARCO_MODEL_DIR is undefined, rerun Cell 10 first.

!python -u eval_msmarco_metrics.py \
  --model_dir "$LATEST_MSMARCO_MODEL_DIR" \
  --data_path data/msmarco_dev.jsonl \
  --max_examples 500 \
  --batch_size 4 \
  --n_docs 5 \
  --num_beams 4 \
  --out_path outputs/msmarco_50k_answerable_predictions.jsonl


Loading model: outputs/rag_baseline_msmarco_50k_answerable/checkpoint-1000
2026-05-08 05:50:39,726 [INFO] Loading tokenizer: outputs/rag_baseline_msmarco_50k_answerable/checkpoint-1000
2026-05-08 05:50:40,449 [INFO] Loading 50k custom retriever...
2026-05-08 05:50:40,449 [INFO] index_path = data/small_nq_index/index.faiss
2026-05-08 05:50:40,450 [INFO] passages_path = data/small_nq_index/passages
2026-05-08 05:50:40,612 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 05:50:40,620 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/rag-sequence-nq/c0d9c6ceda8a69c78091abb7aa734a97b75b89fd/config.json "HTTP/1.1 200 OK"
2026-05-08 05:50:40,753 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/question_encoder_tokenizer/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 05:50:40,761 [INFO] HTTP Request: HEAD https

In [ ]:
#Cell 12 - 3 epochs

!python -u eval_msmarco_metrics.py \
  --model_dir "$LATEST_MSMARCO_MODEL_DIR" \
  --data_path data/msmarco_dev.jsonl \
  --max_examples 1000 \
  --batch_size 4 \
  --n_docs 5 \
  --num_beams 4 \
  --out_path outputs/msmarco_50k_answerable_3ep_predictions.jsonl

Loading model: outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-3000
2026-05-08 08:44:22,785 [INFO] Loading tokenizer: outputs/rag_baseline_msmarco_50k_answerable_3ep/checkpoint-3000
2026-05-08 08:44:23,462 [INFO] Loading 50k custom retriever...
2026-05-08 08:44:23,462 [INFO] index_path = data/small_nq_index/index.faiss
2026-05-08 08:44:23,462 [INFO] passages_path = data/small_nq_index/passages
2026-05-08 08:44:23,754 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 08:44:23,760 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/rag-sequence-nq/c0d9c6ceda8a69c78091abb7aa734a97b75b89fd/config.json "HTTP/1.1 200 OK"
2026-05-08 08:44:24,007 [INFO] HTTP Request: HEAD https://huggingface.co/facebook/rag-sequence-nq/resolve/main/question_encoder_tokenizer/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 08:44:24,012 [INFO] HTTP Request: HE

In [ ]:
# Cell 13 — inspect a few generated MS-MARCO predictions

#!head -n 5 outputs/msmarco_50k_answerable_predictions.jsonl

!head -n 5 outputs/msmarco_50k_answerable_3ep_predictions.jsonl


{"question": ". what is a corporation?", "prediction": "A corporation is a legal entity that has the legal rights and responsibilities of a government.", "references": ["A corporation is a company or group of people authorized to act as a single entity and recognized as such in law."]}
{"question": "why did rachel carson write an obligation to endure", "prediction": "Rachel Carson wrote an obligation to endure because she wanted to inspire others to do the same.", "references": ["Rachel Carson writes The Obligation to Endure because believes that as man tries to eliminate unwanted insects and weeds, however he is actually causing more problems by polluting the environment."]}
{"question": "symptoms of a dying mouse", "prediction": "Nausea, vomiting, diarrhea, fever, and lethargy are the symptoms of a dying mouse.", "references": ["The symptoms of a dying mouse are runny eyes, sneezing, wheezing, shaking, fluctuating body temperature, tiredness, loss of appetite, and dull coat."]}
{"que

In [ ]:
!grep -i "No Answer Present" data/msmarco_train.jsonl | head
!grep -i "No Answer Present" data/msmarco_dev.jsonl | head

!grep -i '"prediction": "No Answer Present' outputs/msmarco_50k_answerable_predictions.jsonl | wc -l

0


## What to report on your poster

Use the numbers printed by Cell 12. A clean table would be:

| Task | Dataset | Metric | Our Result | Paper RAG-Seq |
|---|---|---:|---:|---:|
| Open-Domain QA | NQ | EM | 11.41 | 44.5 |
| Abstractive QA | MS-MARCO | ROUGE-L | your value | 40.8 |
| Abstractive QA | MS-MARCO | BLEU-1 | your value | 44.2 |

Important wording:

> We trained a separate MS-MARCO abstractive QA model using the same stable 50k retriever corpus and evaluated generated answers with BLEU-1 and ROUGE-L.
